## Loading our model

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

local_model_path = r"D:\models\Qwen2.5-0.5B"

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    device_map="cuda",
    dtype="auto",
    trust_remote_code=True,
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    local_files_only=True
)
# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 731.34it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
# Prompt
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate the output
output = pipe(messages)
print(output[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sure, here's a funny joke about chickens:

Why did the chicken go to the doctor?

Because it was feeling a bit "chicken"!


In [3]:
# Apply prompt template
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

<|im_start|>user
Create a funny joke about chickens.<|im_end|>



In [4]:
# Using a high temperature
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why did the chicken cross the road?

To get to the other side… and then *stomp* its feet to get the attention of a squirrel who was trying to steal its eggs!

(And no, the squirrel didn’t get the eggs — it just got *very* confused and started wearing a tiny hat to look more like a chicken.) 🐔🧢


In [5]:
# Using a high top_p
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why did the chicken cross the road?

To get to the other side — and then *sneak* a little egg from the neighbor’s coop!  

(Also, it was just a *hatch* of bad decisions.) 🐔😄


# **Intro to Prompt Engineering**


## The Basic Ingredients of a Prompt


# **Advanced Prompt Engineering**


## Complex Prompt

In [6]:
# Text to summarize which we stole from https://jalammar.github.io/illustrated-transformer/ ;)
text = """In the previous post, we looked at Attention – a ubiquitous method in modern deep learning models. Attention is a concept that helped improve the performance of neural machine translation applications. In this post, we will look at The Transformer – a model that uses attention to boost the speed with which these models can be trained. The Transformer outperforms the Google Neural Machine Translation model in specific tasks. The biggest benefit, however, comes from how The Transformer lends itself to parallelization. It is in fact Google Cloud’s recommendation to use The Transformer as a reference model to use their Cloud TPU offering. So let’s try to break the model apart and look at how it functions.
The Transformer was proposed in the paper Attention is All You Need. A TensorFlow implementation of it is available as a part of the Tensor2Tensor package. Harvard’s NLP group created a guide annotating the paper with PyTorch implementation. In this post, we will attempt to oversimplify things a bit and introduce the concepts one by one to hopefully make it easier to understand to people without in-depth knowledge of the subject matter.
Let’s begin by looking at the model as a single black box. In a machine translation application, it would take a sentence in one language, and output its translation in another.
Popping open that Optimus Prime goodness, we see an encoding component, a decoding component, and connections between them.
The encoding component is a stack of encoders (the paper stacks six of them on top of each other – there’s nothing magical about the number six, one can definitely experiment with other arrangements). The decoding component is a stack of decoders of the same number.
The encoders are all identical in structure (yet they do not share weights). Each one is broken down into two sub-layers:
The encoder’s inputs first flow through a self-attention layer – a layer that helps the encoder look at other words in the input sentence as it encodes a specific word. We’ll look closer at self-attention later in the post.
The outputs of the self-attention layer are fed to a feed-forward neural network. The exact same feed-forward network is independently applied to each position.
The decoder has both those layers, but between them is an attention layer that helps the decoder focus on relevant parts of the input sentence (similar what attention does in seq2seq models).
Now that we’ve seen the major components of the model, let’s start to look at the various vectors/tensors and how they flow between these components to turn the input of a trained model into an output.
As is the case in NLP applications in general, we begin by turning each input word into a vector using an embedding algorithm.
Each word is embedded into a vector of size 512. We'll represent those vectors with these simple boxes.
The embedding only happens in the bottom-most encoder. The abstraction that is common to all the encoders is that they receive a list of vectors each of the size 512 – In the bottom encoder that would be the word embeddings, but in other encoders, it would be the output of the encoder that’s directly below. The size of this list is hyperparameter we can set – basically it would be the length of the longest sentence in our training dataset.
After embedding the words in our input sequence, each of them flows through each of the two layers of the encoder.
Here we begin to see one key property of the Transformer, which is that the word in each position flows through its own path in the encoder. There are dependencies between these paths in the self-attention layer. The feed-forward layer does not have those dependencies, however, and thus the various paths can be executed in parallel while flowing through the feed-forward layer.
Next, we’ll switch up the example to a shorter sentence and we’ll look at what happens in each sub-layer of the encoder.
Now We’re Encoding!
As we’ve mentioned already, an encoder receives a list of vectors as input. It processes this list by passing these vectors into a ‘self-attention’ layer, then into a feed-forward neural network, then sends out the output upwards to the next encoder.
"""

# Prompt components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
# text = "MY TEXT TO SUMMARIZE"  # Replace with your own text to summarize
data = f"Text to summarize: {text}"

# The full prompt - remove and add pieces to view its impact on the generated output
query = persona + instruction + context + data_format + audience + tone + data

In [7]:
messages = [
    {"role": "user", "content": query}
]
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|im_start|>user
You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.
Summarize the key findings of the paper provided.
Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.
Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.
The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.
The tone should be professional and clear.
Text to summarize: In the previous post, we looked at Attention – a ubiquitous method in modern deep learning models. Attention is a concept that helped improve the performance of neural machine translation applications. In this post, we will look at The Transformer – a model that uses attention to boost the speed with which these models can be trained. The Transformer outperforms the Goo

In [8]:
# Generate the output
outputs = pipe(messages)
print(outputs[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Bullet-Point Summary of Method:**

- The Transformer model consists of two main components: an **encoding stack** and a **decoding stack**, each composed of identical layers (e.g., 6 encoders and 6 decoders).
- Each encoder and decoder contains two core sub-layers:
  - **Self-attention layer**: Enables each position in the input sequence to attend to all other positions, allowing contextual understanding of word relationships.
  - **Feed-forward neural network**: Applies the same transformation independently to each position, with no inter-position dependencies.
- **Parallelization advantage**: The feed-forward layers are independent across positions, enabling full parallel computation during training and inference.
- **Input processing**: Words are converted into fixed-dimensional vectors (e.g., 512-dim embeddings) via an embedding layer, applied only at the first encoder.
- **Flow through encoders**: Each word’s vector passes through its own path in each encoder, with self-attentio

## In-Context Learning: Providing Examples

In [9]:
# Use a single example of using the made-up word in a sentence
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

<|im_start|>user
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:<|im_end|>
<|im_start|>assistant
I have a Gigamuru that my uncle gave me as a gift. I love to play it at home.<|im_end|>
<|im_start|>user
To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:<|im_end|>



In [10]:
# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


It's a common mistake—there is no word "screeg" in English that means to swing a sword at something. The correct word for swinging a sword is likely "sweep" or "swipe," and "screech" is a verb that means to make a loud, high-pitched sound, such as a bird or a car tire.

Therefore, a sentence using the word *screech* (a common word, though not "screeg") would be:

"The dragon screeched as it flew past the castle."

But since "screeg" is not a valid or recognized English word, no legitimate sentence can use it in the context you described. It may be a misspelling or a fictional term.

✅ Corrected and accurate response:  
There is no valid English word "screeg" that means to swing a sword. The intended word is likely "screech" (to make a loud noise), not "swing a sword." Thus, a sentence using a real word would be:  
"The sword screeched as it cut through the air."  
(Note: This is a poetic or metaphorical use, not literal—swords don’t screech, but the sound of metal on metal might be des

## Chain Prompting: Breaking up the Problem


In [11]:
# Create name and slogan for a product
product_prompt = [
    {"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}
]
outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Name:** NovaMind  
**Slogan:** *Thinking with you — smarter, faster, always ready.*

**Why it works:**  
"Nova" evokes innovation, new beginnings, and the future — perfect for an AI powered by large language models. "Mind" suggests intelligence, understanding, and human-like reasoning. The slogan emphasizes real-time, intuitive, and adaptive conversation, highlighting both the bot’s capabilities and its user-centered design.


In [12]:
# Based on a name and slogan for a product, generate a sales pitch
sales_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}
]
outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]
print(sales_pitch)

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Introducing NovaMind — thinking with you, smarter, faster, always ready. Your AI partner, built for intuitive, real-time conversations that adapt to you. Future-ready. Human-like. Always on.


# **Reasoning with Generative Models**


## Chain-of-Thought: Think Before Answering


In [13]:
# Answering with chain-of-thought
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."},
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]

# Generate the output
outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The cafeteria started with 23 apples.  
They used 20 apples, so:  
23 - 20 = 3 apples left.  
Then they bought 6 more:  
3 + 6 = 9 apples.  

So, they now have **9 apples**. ✅


## Zero-shot Chain-of-Thought


In [14]:
# Zero-shot Chain-of-Thought
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]

# Generate the output
outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Let's solve this step-by-step:

1. The cafeteria originally had **23 apples**.  
2. They used **20 apples** to make lunch.  
   So, after using 20, the number of apples left is:  
   23 - 20 = **3 apples**  
3. Then, they **bought 6 more apples**.  
   So, adding those to the remaining 3:  
   3 + 6 = **9 apples**

Final answer: **9 apples** ✅


## Tree-of-Thought: Exploring Intermediate Steps


In [15]:
# Zero-shot Chain-of-Thought
zeroshot_tot_prompt = [
    {"role": "user", "content": "Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group. Then all experts will go on to the next step, etc. If any expert realises they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."}
]

In [16]:
# Generate the output
outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Let's begin the expert thinking process step by step. We have three experts — let's call them **Dr. Math**, **Chef Jamie**, and **Lily the Logician** — each taking one step at a time. After each step, they share their thought, and then all proceed to the next step. If any expert realizes they made a mistake at any point, they leave the discussion.

---

### **Step 1: Initial Understanding**

**Dr. Math:**  
I start by understanding the problem: the cafeteria began with 23 apples. They used 20 for lunch, so I subtract 20 from 23. That leaves 3 apples. Then they bought 6 more, so I add 6 to 3. Final total should be 3 + 6 = 9 apples.

**Chef Jamie:**  
I think about it from a practical kitchen perspective. We start with 23 apples. We use 20 — that’s a lot, so we’re left with 3. Then we buy 6 more. So now we have 3 + 6 = 9 apples. That makes sense for a lunch setup.

**Lily the Logician:**  
I’m thinking about the sequence: start with 23, subtract 20 (used), then add 6 (bought). So: 23 – 2

# **Output Verification**

## Providing Examples

In [17]:
# Zero-shot learning: Providing no examples
zeroshot_prompt = [
    {"role": "user", "content": "Create a character profile for an RPG game in JSON format."}
]

# Generate the output
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "character_name": "Kaelen Vireth",
  "race": "Elven",
  "class": "Rogue (Arcane Trickster)",
  "level": 18,
  "background": "Outlander",
  "alignment": "Neutral Good",
  "gender": "Male",
  "age": 245,
  "appearance": {
    "height": "5'10\"",
    "build": "Slim and agile",
    "hair": "Silver-white with faint blue streaks",
    "eyes": "Hazel with a shimmering, almost iridescent quality",
    "notable_features": "A faint glowing rune etched on his left forearm, pulsing with arcane energy"
  },
  "personality": {
    "traits": [
      "Cunning and observant",
      "Loyal to friends and allies",
      "Witty and quick with a sarcastic remark",
      "Deeply empathetic but guarded"
    ],
    "flaws": [
      "Overly trusting of promises",
      "Can be reckless when emotions run high",
      "Struggles with vulnerability"
    ],
    "motivation": "To uncover the truth behind the ancient magic that corrupted his people and restore balance to the world."
  },
  "abilities": {
    "st

In [18]:
# One-shot learning: Providing an example of the output structure
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:

{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [
    {"role": "user", "content": one_shot_template}
]

# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "description": "A rogue scholar who uses ancient texts to predict enemy movements and outwit foes with clever traps.",
  "name": "Elira Voss",
  "armor": "Leather scholar's cloak",
  "weapon": ["ink-stained quill", "scroll of illusions"]
}
